# 15 - Meta-Fusion of Camera + USS Low/High

Notebook 14 showed the camera and USS have **complementary** failure modes
(camera good at tall/vertical, USS good at flat/low), but the naive "trust the
detector" rule captures none of it - fused == detector == 0.936. USS is right
on ~60% of the camera's errors (mostly flat/low), so a learned combiner should
do better.

This fits a tiny **logistic meta-classifier** on the signal features
`[det_is_high, det_conf, has_detection, uss_prob, has_uss]`, calibrated on a
sequence-split and evaluated held-out. No GPU, no re-running the detector - it
reads `fusion_signals.csv` from notebook 14.

Reports meta-fusion vs USS-alone, detector-alone, and the naive rule, plus the
learned weights (which signal it trusts when).

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")

In [ ]:
import json as _json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from vision_uss_research.alignment.fusion import (LogisticFusion, balanced_accuracy,
                                                 rule_fusion, signal_features)
from vision_uss_research.runs import finish_run, new_run_id, resolve_run, start_run

SEED = 0
CALIB_FRAC = 0.6
FUSION_RUN = None      # reports/<run_id> with fusion_signals.csv; None = newest

fus_run = (ARTIFACTS / "reports" / FUSION_RUN) if FUSION_RUN else resolve_run(
    ARTIFACTS / "reports", "fusion", require_file="fusion_signals.csv")
assert fus_run is not None, "run notebook 14 first (fusion_signals.csv)"
sig = pd.read_csv(fus_run / "fusion_signals.csv")
# keep rows that have BOTH signals (the fair meta-fusion set)
sig = sig[sig["uss_bin"].notna() & sig["det_bin"].notna()].reset_index(drop=True)
sig["uss_prob"] = pd.to_numeric(sig["uss_prob"], errors="coerce")
sig["det_conf"] = pd.to_numeric(sig["det_conf"], errors="coerce")

RUN_DIR = ARTIFACTS / "reports" / new_run_id("fusion", "meta")
RUN_DIR.mkdir(parents=True, exist_ok=True)
start_run(RUN_DIR, config={"calib_frac": CALIB_FRAC, "seed": SEED},
          inputs=[f"reports/{fus_run.name}"], repo_root=REPO_DIR)
print(f"fusion signals run: {fus_run.name}")
print(f"sequences (both signals): {len(sig)} "
      f"(low={int((sig['truth']=='low').sum())}, high={int((sig['truth']=='high').sum())})")

## Fit the meta-classifier on a sequence-split, evaluate held-out

In [ ]:
rng = np.random.default_rng(SEED)
ids = sorted(sig["sequence_id"].astype(str).unique())
rng.shuffle(ids)
n_cal = int(CALIB_FRAC * len(ids))
cal_ids, test_ids = set(ids[:n_cal]), set(ids[n_cal:])
cal = sig[sig["sequence_id"].astype(str).isin(cal_ids)]
test = sig[sig["sequence_id"].astype(str).isin(test_ids)]

y_cal = (cal["truth"] == "high").astype(int).values
X_cal = signal_features(cal["det_bin"], cal["det_conf"], cal["uss_prob"])
clf = LogisticFusion(epochs=3000, seed=SEED).fit(X_cal, y_cal)

X_test = signal_features(test["det_bin"], test["det_conf"], test["uss_prob"])
test = test.assign(meta_bin=clf.predict_bin(X_test),
                   naive_bin=rule_fusion(test["det_bin"], test["uss_bin"]))

def report(col):
    return (balanced_accuracy(test[col], test["truth"]),
            (test[col] == test["truth"]).mean())

print(f"HELD-OUT ({len(test)} sequences)     balanced-acc   accuracy")
for name, col in [("USS alone", "uss_bin"), ("Detector alone", "det_bin"),
                  ("Naive fusion (rule)", "naive_bin"),
                  ("META fusion (learned)", "meta_bin")]:
    bacc, acc = report(col)
    print(f"  {name:<22} {bacc:.3f}         {acc:.3f}")

## What the meta-fuser learned, and where it helps

In [ ]:
feat_names = ["det_is_high", "det_conf", "has_det", "uss_prob", "has_uss"]
weights = pd.Series(clf.w, index=feat_names).round(3)
print("learned weights (standardized; sign = pushes toward 'high'):")
display(weights.to_frame("weight"))

# per-class: naive vs meta on the held-out set
def bacc_by(col, g):
    return (g[col] == g["truth"]).mean()
per_class = (test.groupby("target_object")
             .apply(lambda g: pd.Series({
                 "n": len(g),
                 "uss": bacc_by("uss_bin", g),
                 "detector": bacc_by("det_bin", g),
                 "meta": bacc_by("meta_bin", g)}), include_groups=False)
             .round(3))
per_class["meta_gain_vs_det"] = (per_class["meta"] - per_class["detector"]).round(3)
display(per_class.sort_values("meta_gain_vs_det", ascending=False))

# where meta fixes a detector error using USS
det_wrong = test[test["det_bin"] != test["truth"]]
meta_fixes = (det_wrong["meta_bin"] == det_wrong["truth"]).mean() if len(det_wrong) else float("nan")
print(f"\ndetector wrong on {len(det_wrong)} held-out seqs -> meta correct on "
      f"{100*meta_fixes:.1f}% of them (USS complementarity recovered)")

In [ ]:
mb, ma = report("meta_bin"); nb_, na = report("naive_bin")
ub, ua = report("uss_bin"); db, da = report("det_bin")
summary = {
    "n_heldout": int(len(test)),
    "uss_balanced_acc": round(float(ub), 4),
    "detector_balanced_acc": round(float(db), 4),
    "naive_fusion_balanced_acc": round(float(nb_), 4),
    "meta_fusion_balanced_acc": round(float(mb), 4),
    "meta_accuracy": round(float(ma), 4),
    "detector_accuracy": round(float(da), 4),
    "meta_fixes_detector_errors_pct": round(float(meta_fixes), 4),
    "weights": {k: round(float(v), 3) for k, v in zip(feat_names, clf.w)},
}
finish_run(RUN_DIR, summary=summary)
print(_json.dumps(summary, indent=1))
test.to_csv(RUN_DIR / "meta_fusion_heldout.csv", index=False)

## Reading the result

- If **meta > detector > USS** in balanced accuracy, the learned fusion has
  recovered the complementarity the naive rule threw away - the clean
  "fusion beats both" figure for the paper.
- **`meta_gain_vs_det` by class** should be positive exactly where USS is
  strong and the camera weak (flat/low: woodenboard, curbstone) - the meta-fuser
  defers to USS there and trusts the camera elsewhere.
- If meta only ties the detector, that is still an honest result: the camera is
  simply dominant, and the naive "trust the detector" rule is near-optimal - a
  clean statement in itself.

Caveats: small held-out set per class; a logistic combiner is deliberately
simple (interpretable weights) - a stronger combiner could do marginally
better but risks overfitting this single-facility data. This closes the fusion
experiment (`docs/v2_open_set_approach.md`, Phase D).